In [1]:
import numpy as np
import pandas as pd
import os
import torch
import torch.nn as nn
import torch.nn.functional as F


from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.datasets import Planetoid, MovieLens100K

from sklearn.metrics import roc_auc_score


In [2]:
# dataset = Planetoid(root='data/Planetoid', name='Cora')
dataset = MovieLens100K(root='data/ml_100k')
data = dataset[0]
print('data')
eps = 1.0e-6

print('data.edge_attrs.shape = ' + str(data.edge_attrs)) 


data
data.edge_attrs.shape = <bound method BaseData.edge_attrs of HeteroData(
  movie={ x=[1682, 18] },
  user={ x=[943, 24] },
  (user, rates, movie)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
    edge_label_index=[2, 20000],
    edge_label=[20000],
  },
  (movie, rated_by, user)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
  }
)>


In [3]:
def subsample_random_walk(edge_index, nstep):
    nedge = edge_index.shape[1]
    ### what do you have to do?
    ### the thing that you have to consider is the 
    edge_list = []
    idx0 = np.random.randint(0, high=nedge)
    v1 = edge_index[1, idx0]
    edge_list = []
    idx_list = []
    idx_list.append(idx0)
    for idx in range(nstep):
        rw_options = np.where(edge_index[0, :] == v1)[0]
        idx0 = np.random.randint(0, high=rw_options.size)
        v1 = edge_index[1, rw_options[idx0]]
        idx_list.append(rw_options[idx0])
        
        # print(v1)
    # print('len(idx_list) = ' + str(len(idx_list)))
    return np.asarray(idx_list, dtype=np.int64)

def make_train_test_split(idata, frac_train=0.7):
    nedge = idata.shape[0]
    ntrain = np.int64(frac_train * nedge)
    perm = np.random.permutation(nedge)
    return (idata[perm[:ntrain], :], idata[perm[ntrain:], :])

def make_index_shift(idata, sdata, nuser):
    nedge = idata.shape[0]
    idx_data = np.zeros((2 * nedge, 2))
    idx_data[:nedge, :] = 0 + idata
    idx_data[:nedge, 1] += nuser
    ### nake transpose matrix
    idx_data[nedge:, 0] = 0 + idx_data[:nedge, 1]
    idx_data[nedge:, 1] = 0 + idx_data[:nedge, 0]
    scr_data = np.zeros((2 * nedge))
    scr_data[:nedge] = 0 + sdata
    scr_data[nedge:] = 0 + sdata
    return (idx_data, scr_data)
    
    
def get_edge_label(sdata, starget):
    nedge = sdata.shape[0]
    scr_data = np.zeros((2 * nedge))
    for idx in range(nedge):
        if sdata[idx] < starget:
            scr_data[idx] = 0
            scr_data[nedge + idx] = 0
        else:
            scr_data[idx] = 1
            scr_data[nedge + idx] = 1
    return scr_data

    
def make_binary_attribure(edge_attrs, scr_threshold, esp=1.0e-6):
    edge_probs = torch.zeros((edge_attrs.shape[0], 2))
    for idx in range(edge_probs.shape[0]):
         if edge_attrs[idx] < scr_threshold:
             edge_probs[idx, 0] = eps
             edge_probs[idx, 1] = 1.0 - eps
         else:
             edge_probs[idx, 0] = 1.0 - eps
             edge_probs[idx, 1] = eps
    return edge_probs


In [4]:
class SimpleGCN(torch.nn.Module):
    def __init__(self, num_nodes, emb_dim):
        super(SimpleGCN, self).__init__()
        self.emb = nn.Embedding(num_nodes, emb_dim) 
        # GCNConv layer: maps input features (1) to output features (16)
        self.conv1 = GCNConv(in_channels=emb_dim, out_channels=64)
        # Another GCNConv layer: maps 16 features to 2 (e.g., for 2 classes in node classification)
        self.conv2 = GCNConv(in_channels=64, out_channels=64)
        self.conv3 = GCNConv(in_channels=64, out_channels=emb_dim)

    def forward(self, data):
        edge_index, edge_attrs = data.edge_index, data.edge_attr
        x = self.emb.weight
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.conv3(x, edge_index)

        s = torch.sum(x[edge_index[0, :], :] * x[edge_index[1, :], :], dim=1)
        loss = torch.mean(torch.log(1.0 + torch.exp(s)) - torch.mean(edge_attrs[:, 0] * s))
        return (s, x, loss)

In [ ]:
def main():
    print('inside the main function')
    ratings = pd.read_csv('./ml-100k/u.data', sep='\t', names=['user_id', 'movie_id', 'rating', 'timestamp'])
    print(ratings.head())
    idata = ratings.values
    print(idata.shape)
    idata[:, 0] -= 1
    idata[:, 1] -= 1
    nuser = np.max(idata[:, 0]) + 1
    nitem = np.max(idata[:, 1]) + 1
    
    ### now you have to make train-test split
    idata_train, idata_ttest = make_train_test_split(idata, frac_train=0.7)
    scr_threshold = 4.0
    idx_data_train, scr_data_train = make_index_shift(idata_train[:, :2], idata_train[:, 2], nuser)
    edge_index_train = torch.from_numpy(np.transpose(idx_data_train)).long()
    edge_attrs_train = make_binary_attribure(torch.from_numpy(scr_data_train).long(), scr_threshold)
    idx_data_ttest, scr_data_ttest = make_index_shift(idata_ttest[:, :2], idata_ttest[:, 2], nuser)
    edge_index_ttest = torch.from_numpy(np.transpose(idx_data_ttest)).long()
    edge_attrs_ttest = make_binary_attribure(torch.from_numpy(scr_data_ttest).long(), scr_threshold)
    
    
    gdata_train = Data(edge_index=edge_index_train, edge_attr=edge_attrs_train)
    gdata_ttest = Data(edge_index=edge_index_ttest, edge_attr=edge_attrs_ttest)
    
    num_nodes = nuser + nitem
    emb_dim = 16
    model = SimpleGCN(num_nodes, emb_dim)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)


    # loader = GraphSAINTRandomWalkSampler(gdata_train, batch_size=6, walk_length=600, num_steps=5)
    nepoch = 60000
    model.emb.requires_grad_(True)
    model.conv1.requires_grad_(True)
    model.conv2.requires_grad_(True)
    
    edge_index_train_numpy = (edge_index_train + 0).detach().cpu().numpy()
    nstep = 600
    for kepoch in range(nepoch):
        print('kepoch = ' + str(kepoch) + ' of nepoch = ' + str(nepoch))
        edge_mask = subsample_random_walk(edge_index_train_numpy, nstep)
        gdata_batch = Data(edge_index=edge_index_train[:, edge_mask[:]], 
                           edge_attr = edge_attrs_train[edge_mask[:], :])
        # gdata_train.edge_attr = edge_attrs_train[edge_mask[:], :]
        s, x, loss = model(gdata_batch)
        total_loss = 0.0
        opt.zero_grad()
        loss.backward()
        opt.step()
        print('loss = ' + str(loss.item()))
    
    gdata_train.edge_index = edge_index_train
    gdata_train.edge_attr = edge_attrs_train
    strain, _, _ = model(gdata_train)
    sttest, _, _ = model(gdata_ttest)
    
    strain_true = get_edge_label(idata_train[:, 2], scr_threshold)
    sttest_true = get_edge_label(idata_ttest[:, 2], scr_threshold)
    strain_pred = strain.detach().cpu().numpy()
    sttest_pred = sttest.detach().cpu().numpy()
    roc_auc_train = roc_auc_score(strain_true, strain_pred)
    roc_auc_ttest = roc_auc_score(sttest_true, sttest_pred)
    print('roc_auc_train = ' + str(roc_auc_train))
    print('roc_auc_ttest = ' + str(roc_auc_ttest))
    return 0

main()



inside the main function
   user_id  movie_id  rating  timestamp
0      196       242       3  881250949
1      186       302       3  891717742
2       22       377       1  878887116
3      244        51       2  880606923
4      166       346       1  886397596
(100000, 4)
kepoch = 0 of nepoch = 60000
loss = 0.6956467032432556
kepoch = 1 of nepoch = 60000
loss = 0.6844303607940674
kepoch = 2 of nepoch = 60000
loss = 0.7036503553390503
kepoch = 3 of nepoch = 60000
loss = 0.6805658340454102
kepoch = 4 of nepoch = 60000
loss = 0.7038955688476562
kepoch = 5 of nepoch = 60000
loss = 0.6880553364753723
kepoch = 6 of nepoch = 60000
loss = 0.6802999377250671
kepoch = 7 of nepoch = 60000
loss = 0.6752853393554688
kepoch = 8 of nepoch = 60000
loss = 0.690003514289856
kepoch = 9 of nepoch = 60000
loss = 0.6898123025894165
kepoch = 10 of nepoch = 60000
loss = 0.6846871972084045
kepoch = 11 of nepoch = 60000
loss = 0.6893478631973267
kepoch = 12 of nepoch = 60000
loss = 0.6948352456092834
kepoch